# 03 — Clean the Scraped Module Data

**Input:**  `data/raw/module_bank/modules_details.csv`  (one row per module-year, scraped in notebook 02)

**Outputs (written to `data/processed2/`):**
- `modules_clean.csv`      — one row per (undergraduate module, year) with derived features
- `dept_year_summary.csv`  — one row per (department, year), aggregated up

## What this notebook does

1. Pull `dept_code`, `level`, `faculty` out of the module code (e.g. `BEE2030` → dept BEE, level 2, faculty ESE).
2. Compute three useful features: `total_hours`, `contact_ratio`, `assessment_diversity`.
3. **Fixes from v1**:
   - **Renormalise** the three assessment percentages so they sum to 100, instead of throwing the row away.
   - **Add the missing dept→faculty mappings** (PYC, ENG, ANT, ARC, EFP, BEP, MBA, …) so HASS / ESE / HLS isn't half-empty.
   - **Keep undergraduate modules only** (levels 1–4). NSS only surveys undergraduates, so mixing Masters in was unfair.
4. Aggregate up to (dept, year) with a credit-weighted mean.


## 1. Setup

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

# Find the project root: the folder that contains data/raw.
HERE = Path.cwd()
PROJECT_ROOT = HERE if (HERE / 'data' / 'raw').exists() else HERE.parent

RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'module_bank' / 'modules_details.csv'
OUT_DIR  = PROJECT_ROOT / 'data' / 'processed2'
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW_PATH)
print(f'Loaded {len(df):,} module-year rows from {RAW_PATH.name}')

## 2. Pull dept code and level out of the module code

Exeter codes look like `BEE2030`: three letters for the department, then a digit (year of study) or `M` / `P` (postgrad).

In [ ]:
CODE_RE = re.compile(r'^([A-Z]{3})([0-9MP])')

def split_code(code):
    if not isinstance(code, str):
        return None, None
    m = CODE_RE.match(code.strip())
    return (m.group(1), m.group(2)) if m else (None, None)

df[['dept_code', 'level_char']] = df['module_code'].apply(
    lambda c: pd.Series(split_code(c))
)

LEVEL_NAMES = {'1': 'UG Year 1', '2': 'UG Year 2', '3': 'UG Year 3',
               '4': 'UG Year 4', 'M': 'Masters', 'P': 'Postgraduate'}
df['level'] = df['level_char'].map(LEVEL_NAMES)

print('Levels found:')
print(df['level'].value_counts(dropna=False))

## 3. Map every department to its faculty

Three faculties at Exeter:
- **ESE** — Environment, Science & Economy
- **HLS** — Health & Life Sciences
- **HASS** — Humanities, Arts & Social Sciences

v1 of this notebook left ~25% of modules as `Unknown` because the mapping was incomplete.  Below is the full list, including the ones we missed: `PYC`, `ENG`, `ANT`, `ARC`, `EFP`, `BEP`, `BEA`, `BEF`, `BUS`, `MBA`, `EAF`, `JBI`, `HIS`, `SPA`, `CMM`.

In [ ]:
DEPT_TO_FACULTY = {
    # ----- ESE: Environment, Science & Economy -----
    'BEE': 'ESE', 'BEM': 'ESE', 'BEA': 'ESE', 'BEF': 'ESE', 'BEP': 'ESE',
    'BUS': 'ESE', 'MBA': 'ESE', 'JBI': 'ESE',           # business / economics
    'ECM': 'ESE', 'ECMM': 'ESE', 'ENG': 'ESE',          # engineering / computing
    'CMM': 'ESE',                                        # comms maths
    'MTH': 'ESE', 'MTHM': 'ESE',                        # maths
    'PHY': 'ESE', 'PHYM': 'ESE',                        # physics
    'GEO': 'ESE', 'GEOM': 'ESE',                        # geography
    'BIO': 'ESE', 'BIOM': 'ESE',                        # biosciences
    'NSC': 'ESE', 'NSCM': 'ESE',                        # natural sciences
    'RES': 'ESE',                                        # renewable energy
    # ----- HLS: Health & Life Sciences -----
    'CSC': 'HLS',
    'HPD': 'HLS', 'HPDM': 'HLS',
    'PSY': 'HLS', 'PSYM': 'HLS', 'PYC': 'HLS',          # psychology (PYC was missing)
    'SHS': 'HLS', 'SHSM': 'HLS',                        # sport & health
    'MED': 'HLS', 'MEDM': 'HLS', 'MDC': 'HLS',          # medicine
    'NUR': 'HLS',                                        # nursing
    # ----- HASS: Humanities, Arts & Social Sciences -----
    'EAS': 'HASS', 'EASM': 'HASS',                      # English
    'EAF': 'HASS',                                       # media/film
    'HIH': 'HASS', 'HIHM': 'HASS', 'HIS': 'HASS',       # history
    'POL': 'HASS', 'POLM': 'HASS',                      # politics
    'SOC': 'HASS', 'SOCM': 'HASS', 'SPA': 'HASS', 'ANT': 'HASS',
    'ARC': 'HASS',                                       # archaeology
    'LAW': 'HASS', 'LAWM': 'HASS',                      # law
    'PHL': 'HASS', 'PHLM': 'HASS', 'THE': 'HASS', 'THEM': 'HASS',
    'CLA': 'HASS', 'CLAM': 'HASS',                      # classics
    'MLF': 'HASS', 'MLG': 'HASS', 'MLS': 'HASS', 'MLI': 'HASS',
    'DRA': 'HASS', 'DRAM': 'HASS',
    'FCS': 'HASS', 'FCSM': 'HASS',
    'ARA': 'HASS', 'ARAM': 'HASS',
    'EDU': 'HASS', 'EDUM': 'HASS', 'EFP': 'HASS',
    'ART': 'HASS', 'ARTM': 'HASS',
    'LIB': 'HASS',
}

df['faculty'] = df['dept_code'].map(DEPT_TO_FACULTY).fillna('Unknown')

print('Faculty distribution after the expanded mapping:')
print(df['faculty'].value_counts())

still_unknown = sorted(df.loc[df['faculty'] == 'Unknown', 'dept_code'].dropna().unique())
print(f'\nStill unmapped: {still_unknown}')

## 4. Three derived features

- `total_hours`         — scheduled + independent + placement (treat missing as 0 if any are present)
- `contact_ratio`       — share of total study hours that are scheduled (lectures, seminars, labs)
- `assessment_diversity` — how many of {coursework, written exam, practical} contribute (1, 2 or 3)

In [ ]:
hours_cols  = ['scheduled_hours', 'independent_hours', 'placement_hours']
assess_cols = ['coursework_pct', 'written_exam_pct', 'practical_exam_pct']

any_hours = df[hours_cols].notna().any(axis=1)
df['total_hours'] = df[hours_cols].fillna(0).sum(axis=1)
df.loc[~any_hours, 'total_hours'] = np.nan

df['contact_ratio'] = df['scheduled_hours'] / df['total_hours']
df.loc[df['total_hours'].fillna(0) == 0, 'contact_ratio'] = np.nan

df['assessment_diversity'] = (df[assess_cols].fillna(0) > 0).sum(axis=1)
df.loc[df[assess_cols].isna().all(axis=1), 'assessment_diversity'] = np.nan

print(df[['total_hours', 'contact_ratio', 'assessment_diversity']].describe().round(2))

## 5. Renormalise the assessment percentages  *(FIX from v1)*

**v1 rule:** drop the row if the three percentages don't sum to 95–105.  That dropped ~1,600 modules.

**v2 rule:** if at least one channel is populated and the sum is in a reasonable band (10–200), rescale all three so they sum to 100. Only drop rows where the sum is essentially zero — those are genuinely broken.

We keep an `assess_was_renormalised` flag so we can later check the result is robust to dropping the rescaled ones.

In [ ]:
before = len(df)
if 'fetch_ok' in df.columns:
    df = df[df['fetch_ok'].fillna(True) == True].copy()

assess_sum = df[assess_cols].fillna(0).sum(axis=1)
df['assess_was_renormalised'] = (~assess_sum.between(95, 105)) & assess_sum.between(10, 200)

needs_rescale = df['assess_was_renormalised']
df.loc[needs_rescale, assess_cols] = (
    df.loc[needs_rescale, assess_cols].fillna(0)
      .div(assess_sum[needs_rescale], axis=0)
      .mul(100)
)

broken = ~assess_sum.between(10, 200)
df = df[~broken].copy()

# v2.1 fix: a row whose two filled columns sum to 100 has the third channel
# showing as blank because there's no contribution from it. The new STEM page
# layout shows blank cells instead of "0%". Treat them as 0 so the column
# means use the same denominator.
filled_total = df[assess_cols].fillna(0).sum(axis=1)
clean_rows = filled_total.between(99.5, 100.5)
for c in assess_cols:
    df.loc[clean_rows & df[c].isna(), c] = 0.0

print(f'Started with: {before:,} rows')
print(f'  rescaled (sum was off):  {needs_rescale.sum():,}')
print(f'  dropped (no assessment): {broken.sum():,}')
print(f'  blank-cell -> 0 fills:   {(filled_total.between(99.5, 100.5) & df[assess_cols].isna().any(axis=1)).sum():,}  (new in v2.1)')
print(f'  kept:                    {len(df):,}  ({len(df)/before:.1%})')

## 6. Keep undergraduate modules only  *(FIX from v1)*

NSS only surveys undergraduates, so for a fair link between module features and NSS we restrict to UG levels (1–4).

In [ ]:
before = len(df)
df = df[df['level_char'].isin(['1', '2', '3', '4'])].copy()
print(f'UG-only filter: {before:,} -> {len(df):,} rows')
print('\nLevels remaining:')
print(df['level'].value_counts())

## 7. Save modules_clean.csv

In [ ]:
modules_clean_path = OUT_DIR / 'modules_clean.csv'
df.to_csv(modules_clean_path, index=False)
print(f'Wrote: {modules_clean_path}  ({len(df):,} rows)')

## 8. Aggregate up to (department, year)

Each row becomes the **credit-weighted mean** of all UG modules in that dept that year. Credit-weighting means a 30-credit module counts twice as much as a 15-credit one.

In [ ]:
df['credit_weight'] = df['credits'].fillna(df['credits'].median())

def w_mean(values, weights):
    v = pd.Series(values).astype(float)
    w = pd.Series(weights).astype(float)
    ok = v.notna() & w.notna() & (w > 0)
    return (v[ok] * w[ok]).sum() / w[ok].sum() if ok.any() else np.nan

def aggregate(g):
    w = g['credit_weight']
    return pd.Series({
        'n_modules':              len(g),
        'mean_coursework_pct':    w_mean(g['coursework_pct'], w),
        'mean_written_exam_pct':  w_mean(g['written_exam_pct'], w),
        'mean_practical_pct':     w_mean(g['practical_exam_pct'], w),
        'mean_class_size':        w_mean(g['num_students_anticipated'], w),
        'mean_scheduled_hours':   w_mean(g['scheduled_hours'], w),
        'mean_contact_ratio':     w_mean(g['contact_ratio'], w),
        'mean_assess_diversity':  w_mean(g['assessment_diversity'], w),
        'mean_n_summative_items': w_mean(g['n_summative_items'], w),
    })

dept_year = (
    df.groupby(['dept_code', 'academic_year', 'faculty'], dropna=False)
      .apply(aggregate, include_groups=False)
      .reset_index()
)

out = OUT_DIR / 'dept_year_summary.csv'
dept_year.to_csv(out, index=False)
print(f'Wrote: {out}  ({len(dept_year):,} rows)')
dept_year.head()

Next: **`04_process_nss3.ipynb`** — clean the NSS data and write the dept-to-NSS-subject mapping.